# PKG edge tables, 2024 → present

**PNC Treasury Management · Data Science · Payment Knowledge Graph**

This notebook builds the deduplicated edge tables behind PKG customer and counterparty metrics, from the Neo4j staging table
`dsihd01p_dsi.neo4j_payments`. The output goes to database `bdahd01p_dlcdi1_cdi_tm` as external parquet tables partitioned by `month`.

**How to run.** Run all cells. Months already built are skipped, so if the session ends, run it again and it continues where it
stopped. To rebuild a month, set `FORCE = True`, or pass just that month in `RUN_MONTHS`. Each finished month is appended
to `pkg_edge_runlog.csv` next to the notebook.

**Measured cost** (trial on 2025-01): about 9.5 min and 6.9 GB per month. For 2024-01 → 2026-08 that is roughly 5.5 h and
220 GB (660 GB on HDFS at replication 3).

## Output tables

| Table | Grain | Columns |
|---|---|---|
| `pkg_edge_daily` | one row per `(txn_date, src, dst, category)` | `txn_date, src_id, src_kind, dst_id, dst_kind, category, cpty_id_src, is_self, amount, n_txn, n_non_usd` |
| `pkg_edge_monthly` | one row per `(src, dst, category)` per month | same keys without `txn_date`, plus `active_days, first_dt, last_dt` |
| `pkg_dedup_5c6c` | one row per dropped 6C duplicate | `trans_id_drop, trans_id_keep, tier, dt_drop, dt_keep, acct, amount, cpty_id_drop, cpty_id_keep` |
| `pkg_cpty_month` | counterparty × name × FI × type × category per month | `cpty_id, cpty_id_src, category, cpty_type, cpty_name, fi, n_txn, amount` |
| `pkg_cpty_dim` | one row per counterparty id, over all months | dominant name / FI / type, `name_key`, `is_business`, `name_is_fi`, `is_placeholder`, activity span, `is_payee` / `is_payer` |
| `pkg_cpty_entity_map` | `cpty_id → ent_l0 / ent_l1 / ent_l2` | `rule` = identity · crosswalk_6c_5c · business_name |

**Column semantics.**
- `src_kind` / `dst_kind`: `CUST` means a PNC customer, identified by `mdm_id`; `CPTY` means an external counterparty.
- `amount`: the sum of `|trans_amt|`, DECIMAL(18,2).
- `n_non_usd`: the number of rows whose `trans_currency` isn't USD. Their amounts are summed as-is, pending confirmation that `trans_amt` is USD-converted.

## Rules and why

| Rule | Reason |
|---|---|
| Debit card (8B/8C) excluded | about 45% of rows but 0.5% of dollars; outside the B2B scope |
| 5C/6C deduplication: drop the 6C twin, keep 5C | 6C (P2P feed) re-records about 21% of its rows from 5C (PRT feed), same account, amount, date and name (names agree on 99.9%). 5C carries the counterparty bank; 6C never does |
| Counterparty id = `unq_cpty_id` for 3B, 3C, 7B, 9B; `unq_cpty_acct_id` for all other categories | for 3B/3C, `unq_cpty_acct_id` is only the originator's 8-digit bank routing number (thousands of originators collapse into one); 7B and 9B have no account id. Elsewhere the account id is stable across name spellings and links inbound with outbound. `cpty_id_src` = A/B records which was used |
| Customers keyed by current `mdm_id` (not `mdm_id_*_orig`) | a stable entity after MDM merges |
| Self-transfers kept, flagged `is_self` | same `mdm_id` on both sides; up to 34% of internal wires. Exclude from graph metrics, keep for liquidity/sweep analysis |
| Entity levels | `ent_l0` = raw id; `ent_l1` = 6C P2P ids folded into their 5C id (dedup pairs, purity ≥ 0.8); `ent_l2` = `ent_l1` + business-like names merged on a normalized key. **Never merged:** person-like names, placeholder names ("BANK ACCOUNT NAME"), and bank-like names (where the name field holds the bank, not the beneficiary). On the trial month, `ent_l2` changed per-customer breadth or concentration for under 1% of customers |

## Open questions with the data owners
- The construction of `unq_cpty_id`.
- The meaning of `np_key` and `mdm_id_*_orig`.
- Whether `trans_amt` is USD-converted for non-USD rows.
- The bank-of-first-deposit routing number for outbound checks (7B).

In [ ]:
# ---- Config -------------------------------------------------------------------------------
import calendar, datetime as dt
def month_range(a, b):
    y, m = map(int, a.split("-")); y2, m2 = map(int, b.split("-")); out = []
    while (y, m) <= (y2, m2):
        out.append(f"{y:04d}-{m:02d}"); y, m = (y + 1, 1) if m == 12 else (y, m + 1)
    return out
_t = dt.date.today().replace(day=1) - dt.timedelta(days=1)
LAST_COMPLETE = f"{_t.year:04d}-{_t.month:02d}"

SRC_TABLE  = "dsihd01p_dsi.neo4j_payments"
TGT_DB     = "bdahd01p_dlcdi1_cdi_tm"
PREFIX     = "pkg_"

RUN_MONTHS = month_range("2024-01", LAST_COMPLETE)   # e.g. ["2025-01"] to (re)build one month
FORCE      = False                                   # True = rebuild months already written

EXCLUDE_CODES = {"8B", "8C"}                         # debit card
B_ID_CODES    = {"3B", "3C", "7B", "8B", "8C", "9B"} # counterparty id = unq_cpty_id here; unq_cpty_acct_id elsewhere
KEEP_CAT, DROP_CAT = "5C.RTP_PRT_CPTY_PAYS", "6C.RTP_P2P_CPTY_PAYS"

XW_MIN_PURITY   = 0.8     # 6C→5C crosswalk: min share of a 6C id's pairs pointing at its top 5C id
NAME_KEY_MINLEN = 4       # ent_l2: shortest normalized business name allowed to merge
SENS_MONTHS     = 3       # sensitivity checks use the latest N built months (breadth, concentration, churn)

FILES = {"edge_daily": 8, "edge_monthly": 4, "cpty_month": 4}   # parquet files per month (~150–350 MB each)
CODEC = "snappy"
SHUFFLE_PARTITIONS = 400
BUILD_DEDUP, BUILD_DIM_MAP, RUN_SENSITIVITY = True, True, True

In [ ]:
# ---- Setup. NumPy shim: Cloudera SPARK3 PySpark predates NumPy 1.24/2.0 aliases used by toPandas()
import numpy as np, warnings
for _alias, _target in {"object0": np.object_, "bool8": np.bool_, "int0": np.intp, "uint0": np.uintp}.items():
    if not hasattr(np, _alias):
        setattr(np, _alias, _target)
if "bool" not in np.__dict__:
    np.bool = np.bool_
warnings.filterwarnings("ignore", category=ResourceWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning, module=r"pyspark.*")

from pyspark.sql import SparkSession, Window, functions as F
from IPython.display import display
from urllib.parse import unquote
from contextlib import contextmanager
import pandas as pd, time

spark = SparkSession.builder.getOrCreate()
spark.conf.set("spark.sql.shuffle.partitions", str(SHUFFLE_PARTITIONS))
print("Spark", spark.version, "| months requested:", RUN_MONTHS[0], "→", RUN_MONTHS[-1], f"({len(RUN_MONTHS)})")
pd.set_option("display.max_rows", 300); pd.set_option("display.max_columns", 60); pd.set_option("display.width", 250)

def show(pdf, title, pct=(), money=(), gb=()):
    print(f"\n■ {title}   [{len(pdf):,} rows]")
    fmt = {}
    for c in pdf.columns:
        if c in pct:                                fmt[c] = "{:.2%}"
        elif c in money:                            fmt[c] = "${:,.0f}"
        elif c in gb:                               fmt[c] = "{:,.2f}"
        elif pd.api.types.is_integer_dtype(pdf[c]): fmt[c] = "{:,}"
        elif pd.api.types.is_float_dtype(pdf[c]):   fmt[c] = "{:,.2f}"
    sty = pdf.style.format(fmt, na_rep="—")
    try:    sty = sty.hide(axis="index")
    except AttributeError: sty = sty.hide_index()     # pandas < 1.4
    display(sty)

def nz(c):
    """String, trimmed, blank → NULL."""
    s = F.trim(F.col(c).cast("string"))
    return F.when(s == "", F.lit(None)).otherwise(s)

TIMINGS = []
@contextmanager
def timed(stage, month=None):
    t0 = time.time(); yield
    s = time.time() - t0
    TIMINGS.append({"month": month, "stage": stage, "seconds": s})
    print(f"  ⏱ {month or '':<8} {stage}: {s/60:,.1f} min")

HCONF = spark._jsc.hadoopConfiguration()
JPath = spark._jvm.org.apache.hadoop.fs.Path
fs_for = lambda p: JPath(p).getFileSystem(HCONF)
exists = lambda p: fs_for(p).exists(JPath(p))
def month_bounds(m):
    y, mo = map(int, m.split("-"))
    return f"{m}-01", f"{m}-{calendar.monthrange(y, mo)[1]:02d}"
code_of = lambda c: c.split(".")[0]
GB = 1e9

## §0 Source categories, target location, table helpers

In [ ]:
CATS = set()
for r in spark.sql(f"SHOW PARTITIONS {SRC_TABLE}").collect():          # metastore only, no data scanned
    kv = {k.lower(): unquote(v) for k, v in (p.split("=", 1) for p in r[0].split("/"))}
    CATS.add(kv["category"])
KEEP_CATS = sorted(c for c in CATS if code_of(c) not in EXCLUDE_CODES)
assert KEEP_CAT in CATS and DROP_CAT in CATS, "5C/6C category names not found"
print(f"{len(KEEP_CATS)} categories kept; excluded: {sorted(c for c in CATS if code_of(c) in EXCLUDE_CODES)}")

SRC_COLS = {c.lower() for c in spark.table(SRC_TABLE).columns}
NEED = {"trans_id", "trans_dt", "trans_amt", "category", "mdm_id_pays", "mdm_id_receives", "pnc_dep_acct_receives",
        "unq_cpty_acct_id", "unq_cpty_id", "cpty_name", "cpty_type", "cpty_fin_entity_name", "trans_currency"}
assert not (NEED - SRC_COLS), f"source is missing {NEED - SRC_COLS}"

DB_LOC = [r[1] for r in spark.sql(f"DESCRIBE DATABASE {TGT_DB}").collect() if r[0].lower() == "location"][0].rstrip("/")
KEYS_T = ["dedup_5c6c", "edge_daily", "edge_monthly", "cpty_month", "cpty_dim", "cpty_entity_map"]
T = {k: f"{TGT_DB}.{PREFIX}{k}" for k in KEYS_T}
P = {k: f"{DB_LOC}/{PREFIX}{k}" for k in KEYS_T}
REPL = int(fs_for(DB_LOC).getDefaultReplication(JPath(DB_LOC)))
print("target:", DB_LOC, "| replication", REPL)

def ensure_table(key, schema, partitioned):
    cols = ", ".join(f"`{f.name}` {f.dataType.simpleString()}" for f in schema.fields if f.name != "month")
    part = " PARTITIONED BY (month STRING)" if partitioned else ""
    spark.sql(f"CREATE EXTERNAL TABLE IF NOT EXISTS {T[key]} ({cols}){part} STORED AS PARQUET LOCATION '{P[key]}'")

def add_partition(key, m):
    spark.sql(f"ALTER TABLE {T[key]} ADD IF NOT EXISTS PARTITION (month='{m}') LOCATION '{P[key]}/month={m}'")

def month_done(key, m):
    return exists(f"{P[key]}/month={m}/_SUCCESS")

def write_month(df, key, m, part_col, sort_cols):
    """One month = one directory, overwritten atomically by Spark; sorted inside files for id compression."""
    (df.repartition(FILES[key], part_col).sortWithinPartitions(*sort_cols)
       .write.mode("overwrite").option("compression", CODEC).parquet(f"{P[key]}/month={m}"))
    ensure_table(key, df.schema, True); add_partition(key, m)

def write_full(df, key):
    df.write.mode("overwrite").option("compression", CODEC).parquet(P[key])
    ensure_table(key, df.schema, False); spark.catalog.refreshTable(T[key])

def size_of(path):
    if not exists(path): return 0, 0
    c = fs_for(path).getContentSummary(JPath(path))
    return c.getLength(), c.getFileCount()

## §1 5C/6C dedup map: one range job for every month still to build

The key is `(date, receiving account, amount, upper-cased cpty_name)`, with one-to-one pairing (the k-th 5C row pairs with the k-th 6C row).
- **Tier 1:** same day.
- **Tier 2:** the 6C row is stamped one day later. This was episodic in Oct–Nov 2024.

The job reads one extra day before the first month, so tier-2 pairs at month edges are found.

In [ ]:
def dedup_pairs(lo, hi):
    lo_ext = (dt.date.fromisoformat(lo) - dt.timedelta(days=1)).isoformat()
    r = (spark.table(SRC_TABLE)
           .filter(F.col("trans_dt").between(lo_ext, hi) & F.col("category").isin(KEEP_CAT, DROP_CAT))
           .select(nz("trans_id").alias("trans_id"), F.to_date("trans_dt").alias("dt"), nz("category").alias("category"),
                   nz("pnc_dep_acct_receives").alias("acct"), F.abs(F.col("trans_amt").cast("decimal(18,2)")).alias("amt"),
                   F.upper(F.regexp_replace(nz("cpty_name"), r"\s+", " ")).alias("nm"), nz("unq_cpty_acct_id").alias("cid"))
           .filter(F.col("acct").isNotNull() & F.col("nm").isNotNull())
           .withColumn("dt0", F.col("dt")))
    A, B = r.filter(F.col("category") == KEEP_CAT), r.filter(F.col("category") == DROP_CAT)
    GK = ["dt", "acct", "amt", "nm"]
    def ranked(df, s):
        w = Window.partitionBy(*GK).orderBy("trans_id")
        return df.withColumn("rn", F.row_number().over(w)).select(
            *GK, "rn", *[F.col(c).alias(f"{c}_{s}") for c in ("trans_id", "dt0", "cid")])
    t1 = ranked(A, "a").join(ranked(B, "b"), GK + ["rn"]).withColumn("tier", F.lit(1))
    A_left = A.join(t1.select(F.col("trans_id_a").alias("trans_id")), "trans_id", "left_anti")
    B_left = B.join(t1.select(F.col("trans_id_b").alias("trans_id")), "trans_id", "left_anti")
    t2 = (ranked(A_left, "a").join(ranked(B_left.withColumn("dt", F.date_sub("dt", 1)), "b"), GK + ["rn"])
            .withColumn("tier", F.lit(2)))
    return (t1.unionByName(t2).filter(F.col("dt0_b") >= F.lit(lo).cast("date"))
              .select(F.col("trans_id_b").alias("trans_id_drop"), F.col("trans_id_a").alias("trans_id_keep"), "tier",
                      F.col("dt0_b").alias("dt_drop"), F.col("dt0_a").alias("dt_keep"), "acct",
                      F.col("amt").alias("amount"), F.col("cid_b").alias("cpty_id_drop"), F.col("cid_a").alias("cpty_id_keep"),
                      F.date_format("dt0_b", "yyyy-MM").alias("month")))

if BUILD_DEDUP:
    todo = [m for m in RUN_MONTHS if FORCE or not exists(f"{P['dedup_5c6c']}/month={m}")]
    if todo:
        lo, hi = month_bounds(min(todo))[0], month_bounds(max(todo))[1]
        with timed(f"dedup range {min(todo)}..{max(todo)} ({len(todo)} months)"):
            pairs = dedup_pairs(lo, hi).filter(F.col("month").isin(todo))
            _prev = spark.conf.get("spark.sql.sources.partitionOverwriteMode", "static")
            spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")    # replace only these months
            (pairs.repartition("month").write.mode("overwrite").option("compression", CODEC)
                  .partitionBy("month").parquet(P["dedup_5c6c"]))
            spark.conf.set("spark.sql.sources.partitionOverwriteMode", _prev)
            ensure_table("dedup_5c6c", pairs.schema, True)
            for m in todo:
                if exists(f"{P['dedup_5c6c']}/month={m}"): add_partition("dedup_5c6c", m)
    else:
        print("dedup: every requested month already present")
    ds = (spark.read.parquet(P["dedup_5c6c"]).filter(F.col("month").isin(RUN_MONTHS))
               .groupBy("month").pivot("tier", [1, 2]).count().orderBy("month").toPandas().fillna(0))
    ds.columns = ["month", "tier1_pairs", "tier2_pairs"]
    show(ds.astype({"tier1_pairs": "int64", "tier2_pairs": "int64"}), "6C rows flagged as duplicates, per month")

## §2 Monthly build: daily edges → monthly edges → counterparty-month

Each month ends with an **exact reconciliation**: the month's source rows and dollars (after dedup) must equal the daily
edges' `n_txn` and `amount`, and those must equal the monthly roll-up's. A mismatch stops the run.

In [ ]:
DEDUP_EXISTS = exists(P["dedup_5c6c"])
KEYS = ["txn_date", "src_id", "src_kind", "dst_id", "dst_kind", "category", "cpty_id_src", "is_self"]

def source_rows(m):
    lo, hi = month_bounds(m)
    s = spark.table(SRC_TABLE).filter(F.col("trans_dt").between(lo, hi) & F.col("category").isin(KEEP_CATS))
    use_b = F.split(nz("category"), r"\.").getItem(0).isin(list(B_ID_CODES))
    A, B  = nz("unq_cpty_acct_id"), nz("unq_cpty_id")
    cid     = F.when(use_b, F.coalesce(B, A)).otherwise(F.coalesce(A, B))
    cid_src = (F.when(use_b & B.isNotNull(), "B").when(use_b & A.isNotNull(), "A")
                .when(A.isNotNull(), "A").when(B.isNotNull(), "B"))
    ccy = F.upper(nz("trans_currency"))
    rows = s.select(nz("trans_id").alias("trans_id"), F.to_date("trans_dt").alias("txn_date"), nz("category").alias("category"),
                    nz("mdm_id_pays").alias("mp"), nz("mdm_id_receives").alias("mr"),
                    cid.alias("cid"), cid_src.alias("cid_src"),
                    F.abs(F.col("trans_amt").cast("decimal(18,2)")).alias("amt"),
                    (ccy.isNotNull() & (ccy != "USD")).cast("int").alias("non_usd"),
                    nz("cpty_name").alias("cpty_name"), nz("cpty_type").alias("cpty_type"),
                    nz("cpty_fin_entity_name").alias("fi"))
    if DEDUP_EXISTS:
        dd = spark.read.parquet(P["dedup_5c6c"]).filter(F.col("month") == m).select(F.col("trans_id_drop").alias("trans_id"))
        rows = (rows.filter(F.col("category") != DROP_CAT)
                    .unionByName(rows.filter(F.col("category") == DROP_CAT).join(dd, "trans_id", "left_anti")))
    internal = F.col("mp").isNotNull() & F.col("mr").isNotNull()
    return (rows
        .withColumn("src_kind", F.when(F.col("mp").isNotNull(), "CUST").otherwise("CPTY"))
        .withColumn("src_id",   F.when(F.col("mp").isNotNull(), F.col("mp")).otherwise(F.col("cid")))
        .withColumn("dst_kind", F.when(F.col("mr").isNotNull(), "CUST").otherwise("CPTY"))
        .withColumn("dst_id",   F.when(F.col("mr").isNotNull(), F.col("mr")).otherwise(F.col("cid")))
        .withColumn("cpty_id_src", F.when(internal, F.lit(None).cast("string")).otherwise(F.col("cid_src")))
        .withColumn("is_self", internal & (F.col("mp") == F.col("mr"))))

MONTH_LOG = []
for m in RUN_MONTHS:
    if not FORCE and all(month_done(k, m) for k in ("edge_daily", "edge_monthly", "cpty_month")):
        print(f"{m}: already built, skipped"); continue
    t_month = time.time()
    rows = source_rows(m)

    with timed("daily edges", m):
        daily = rows.groupBy(*KEYS).agg(F.sum("amt").cast("decimal(18,2)").alias("amount"),
                                        F.count("*").cast("int").alias("n_txn"),
                                        F.sum("non_usd").cast("int").alias("n_non_usd"))
        write_month(daily, "edge_daily", m, "src_id", ["src_id", "dst_id", "txn_date"])

    d = spark.read.parquet(f"{P['edge_daily']}/month={m}")
    with timed("monthly edges", m):
        monthly = (d.groupBy(*KEYS[1:]).agg(
                      F.sum("amount").cast("decimal(18,2)").alias("amount"), F.sum("n_txn").cast("int").alias("n_txn"),
                      F.sum("n_non_usd").cast("int").alias("n_non_usd"), F.count("*").cast("int").alias("active_days"),
                      F.min("txn_date").alias("first_dt"), F.max("txn_date").alias("last_dt")))
        write_month(monthly, "edge_monthly", m, "src_id", ["src_id", "dst_id"])

    with timed("counterparty-month", m):
        cm = (rows.filter(F.col("cpty_id_src").isNotNull())
                  .groupBy(F.col("cid").alias("cpty_id"), "cpty_id_src", "category", "cpty_type", "cpty_name", "fi")
                  .agg(F.count("*").alias("n_txn"), F.sum("amt").cast("decimal(22,2)").alias("amount")))
        write_month(cm, "cpty_month", m, "cpty_id", ["cpty_id"])

    with timed("reconciliation", m):
        src_t = rows.groupBy("category").agg(F.count("*").alias("src_rows"), F.sum("amt").alias("src_amt"),
                                             F.sum("non_usd").alias("non_usd_rows"),
                                             F.sum(F.col("is_self").cast("int")).alias("self_rows")).toPandas()
        edg_t = d.groupBy("category").agg(F.sum("n_txn").alias("edge_rows"), F.sum("amount").alias("edge_amt"),
                                          F.count("*").alias("daily_edges"),
                                          F.sum((F.col("src_id").isNull() | F.col("dst_id").isNull()).cast("long")).alias("null_id_edges")).toPandas()
        mon_t = spark.read.parquet(f"{P['edge_monthly']}/month={m}").groupBy("category").agg(
                    F.count("*").alias("monthly_edges"), F.sum("amount").alias("mon_amt")).toPandas()
        q = src_t.merge(edg_t, on="category", how="outer").merge(mon_t, on="category", how="outer")
        q["month"] = m
        q["ok"] = (q.src_rows == q.edge_rows) & (q.src_amt == q.edge_amt) & (q.edge_amt == q.mon_amt)
    MONTH_LOG.append(q)
    secs = time.time() - t_month
    with open("pkg_edge_runlog.csv", "a") as f:
        f.write(f"{dt.datetime.now().isoformat(timespec='seconds')},{m},{secs:.0f},{int(q.src_rows.sum())},{bool(q.ok.all())}\n")
    print(f"{m}: done in {secs/60:,.1f} min | reconciliation {'OK' if q.ok.all() else 'MISMATCH ❌'}")
    if not q.ok.all():
        show(q[~q.ok][["month", "category", "src_rows", "edge_rows", "src_amt", "edge_amt", "mon_amt"]], "❌ RECONCILIATION MISMATCH")
        raise AssertionError(f"{m}: source and edge totals differ; fix before continuing")

## §3 QA: this run

In [ ]:
if MONTH_LOG:
    Q = pd.concat(MONTH_LOG, ignore_index=True)
    for c in ("src_amt", "edge_amt", "mon_amt"): Q[c] = Q[c].astype(float)
    per_m = (Q.groupby("month").agg(src_rows=("src_rows", "sum"), src_usd=("src_amt", "sum"),
                                    daily_edges=("daily_edges", "sum"), monthly_edges=("monthly_edges", "sum"),
                                    null_id_edges=("null_id_edges", "sum"), ok=("ok", "all")).reset_index())
    show(per_m.astype({"src_rows": "int64", "daily_edges": "int64", "monthly_edges": "int64", "null_id_edges": "int64"}),
         "Per month: rows, dollars, edges, reconciliation", money=("src_usd",))
    last = Q[Q.month == Q.month.max()].copy()
    last["code"] = last.category.map(code_of)
    last["daily_vs_rows"]   = last.daily_edges / last.src_rows
    last["monthly_vs_rows"] = last.monthly_edges / last.src_rows
    last["self_share"]      = last.self_rows / last.src_rows
    last["non_usd_share"]   = last.non_usd_rows / last.src_rows
    show(last[["code", "src_rows", "src_amt", "daily_edges", "daily_vs_rows", "monthly_edges", "monthly_vs_rows",
               "null_id_edges", "self_share", "non_usd_share"]].astype({"src_rows": "int64", "daily_edges": "int64",
               "monthly_edges": "int64", "null_id_edges": "int64"}),
         f"Per category, latest built month ({Q.month.max()})",
         pct=("daily_vs_rows", "monthly_vs_rows", "self_share", "non_usd_share"), money=("src_amt",))
else:
    print("nothing built in this run (all requested months already present)")

## §4 Storage and time

In [ ]:
built = [m for m in RUN_MONTHS if month_done("edge_daily", m)]
st = []
for k in ("dedup_5c6c", "edge_daily", "edge_monthly", "cpty_month"):
    sizes = [size_of(f"{P[k]}/month={m}") for m in built]
    tot_b = sum(s[0] for s in sizes)
    st.append({"table": T[k], "months": len(built), "GB_total": tot_b / GB,
               "GB_per_month": tot_b / max(len(built), 1) / GB, f"GB_on_HDFS_x{REPL}": tot_b * REPL / GB,
               "files_per_month": sum(s[1] for s in sizes) / max(len(built), 1)})
S = pd.DataFrame(st)
S = pd.concat([S, pd.DataFrame([{"table": "TOTAL", "months": len(built),
                                  **{c: S[c].sum() for c in S.columns if c not in ("table", "months")}}])], ignore_index=True)
show(S, "Storage of the requested months", gb=("GB_total", "GB_per_month", f"GB_on_HDFS_x{REPL}", "files_per_month"))
try:
    q = fs_for(DB_LOC).getQuotaUsage(JPath(DB_LOC))
    if q.getSpaceQuota() >= 0:
        print(f"quota at {DB_LOC}: {(q.getSpaceQuota() - q.getSpaceConsumed())/GB/1000:,.2f} TB remaining (replicated)")
except Exception as e:
    print("quota lookup skipped:", str(e)[:120])

if TIMINGS:
    TM = pd.DataFrame(TIMINGS)
    per = TM[TM.month.notna()].groupby("month").seconds.sum() / 60
    remaining = len([m for m in RUN_MONTHS if not month_done("edge_daily", m)])
    show(pd.DataFrame([{"months_built_this_run": len(per), "min_per_month": per.min() if len(per) else None,
                        "mean_per_month": per.mean() if len(per) else None, "max_per_month": per.max() if len(per) else None,
                        "dedup_min": TM[TM.stage.str.startswith("dedup")].seconds.sum() / 60,
                        "months_remaining": remaining}]),
         "Run timing (minutes)")

## §5 Counterparty dimension and entity map (rebuilt from every built month)

`name_key` is the upper-cased name, reduced to letters and digits, with legal suffixes removed (INC, LLC, CO, CORP, N.A., …).
Business-name merging (`ent_l2`) **skips** three kinds of name:
- **person-like names:** no business token in the name;
- **placeholders:** `PLACEHOLDER_KEYS`;
- **bank-like names** (`FI_LIKE`): the name field holds the bank, not the beneficiary.

The largest-merges table is a sanity check, not a result. Names like CIGNA or COINBASE merging thousands of ids is expected: one company behind many ids.

In [ ]:
BIZ = (r"\b(INC|INCORPORATED|LLC|L L C|CO|CORP|CORPORATION|COMPANY|LTD|LIMITED|LP|LLP|PC|PLLC|PA|NA|BANK|TRUST|FUND|"
       r"FOUNDATION|ASSOC|ASSOCIATION|SERVICES?|GROUP|HOLDINGS|PARTNERS|UNIVERSITY|COLLEGE|SCHOOL|DISTRICT|CITY|COUNTY|"
       r"STATE|TOWNSHIP|BOROUGH|HOSPITAL|CENTER|CHURCH|DEPT|DEPARTMENT|TREASURER|INSURANCE|SUPPLY|ENTERPRISES|"
       r"INTERNATIONAL|AUTHORITY|SYSTEMS|SOLUTIONS|TECHNOLOGIES|MANAGEMENT|PROPERTIES|REALTY|CONSTRUCTION)\b")
LEGAL = r"\b(N A|INC|INCORPORATED|LLC|L L C|CO|CORP|CORPORATION|COMPANY|LTD|LIMITED|LP|LLP|PC|PLLC|PA|NA|THE)\b"
FI_LIKE = r"\b(BANK|BANCORP|BANCSHARES|N A|NATIONAL ASSOCIATION|CREDIT UNION|FCU|SAVINGS|TRUST COMPANY)\b"
PLACEHOLDER_KEYS = ["BANK ACCOUNT NAME", "ACCOUNT NAME", "UNKNOWN", "UNKNOWN PAYEE", "NONE", "NULL", "N A", "NA",
                    "TEST", "PAYEE", "VENDOR", "CUSTOMER", "ACCOUNT", "CASH", "CHECK", "DEPOSIT", "TRANSFER",
                    "PAYMENT", "REFUND", "MISC", "MISCELLANEOUS", "VARIOUS", "NAME"]

clean = lambda c: F.trim(F.regexp_replace(F.regexp_replace(F.upper(c), "[^A-Z0-9 ]", " "), r"\s+", " "))
def raw_key(c):
    return F.trim(F.regexp_replace(F.regexp_replace(clean(c), LEGAL, " "), r"\s+", " "))

if BUILD_DIM_MAP:
    with timed("counterparty dimension"):
        cm_all = spark.read.parquet(P["cpty_month"])
        def dominant(col):
            w = Window.partitionBy("cpty_id").orderBy(F.desc("n"), F.col(col))
            return (cm_all.filter(F.col(col).isNotNull()).groupBy("cpty_id", col).agg(F.sum("n_txn").alias("n"))
                          .withColumn("r", F.row_number().over(w)).filter("r = 1").select("cpty_id", col))
        nvar = cm_all.filter(F.col("cpty_name").isNotNull()).groupBy("cpty_id").agg(F.countDistinct("cpty_name").alias("n_name_variants"))
        letter = F.regexp_extract(F.split("category", r"\.").getItem(0), "([ABC])$", 1)
        base = cm_all.groupBy("cpty_id").agg(
                   F.max("cpty_id_src").alias("cpty_id_src"), F.sum("n_txn").alias("n_txn"),
                   F.sum("amount").cast("decimal(22,2)").alias("amount"),
                   F.min("month").alias("first_month"), F.max("month").alias("last_month"),
                   F.countDistinct("month").alias("active_months"),
                   F.concat_ws(",", F.sort_array(F.collect_set(F.split("category", r"\.").getItem(0)))).alias("categories"),
                   F.max((letter == "B").cast("int")).cast("boolean").alias("is_payee"),
                   F.max((letter == "C").cast("int")).cast("boolean").alias("is_payer"))
        rk = raw_key(F.col("cpty_name"))
        dim = (base.join(dominant("cpty_name"), "cpty_id", "left").join(dominant("fi"), "cpty_id", "left")
                   .join(dominant("cpty_type"), "cpty_id", "left").join(nvar, "cpty_id", "left")
                   .withColumn("is_business",    F.coalesce(clean(F.col("cpty_name")).rlike(BIZ), F.lit(False)))
                   .withColumn("name_is_fi",     F.coalesce(clean(F.col("cpty_name")).rlike(FI_LIKE), F.lit(False)))
                   .withColumn("is_placeholder", F.coalesce(rk.isin(PLACEHOLDER_KEYS) | clean(F.col("cpty_name")).isin(PLACEHOLDER_KEYS), F.lit(False)))
                   .withColumn("name_key", F.when(~F.col("is_placeholder") & (F.length(rk) >= NAME_KEY_MINLEN), rk)))
        write_full(dim, "cpty_dim")

    with timed("entity map"):
        dim = spark.table(T["cpty_dim"])
        xw = (spark.read.parquet(P["dedup_5c6c"]).filter(F.col("cpty_id_drop").isNotNull() & F.col("cpty_id_keep").isNotNull())
                   .groupBy("cpty_id_drop", "cpty_id_keep").agg(F.count("*").alias("pairs")))
        wx = Window.partitionBy("cpty_id_drop").orderBy(F.desc("pairs"), "cpty_id_keep")
        xw = (xw.withColumn("tot", F.sum("pairs").over(Window.partitionBy("cpty_id_drop")))
                .withColumn("r", F.row_number().over(wx)).filter("r = 1")
                .filter(F.col("pairs") / F.col("tot") >= XW_MIN_PURITY)
                .select(F.col("cpty_id_drop").alias("cpty_id"), F.col("cpty_id_keep").alias("xw_to")))
        emap = (dim.select("cpty_id", "cpty_id_src").join(xw, "cpty_id", "left")
                   .withColumn("ent_l0", F.col("cpty_id"))
                   .withColumn("ent_l1", F.coalesce("xw_to", "cpty_id")))
        tgt = dim.select(F.col("cpty_id").alias("ent_l1"), F.col("name_key").alias("t_key"),
                         F.col("is_business").alias("t_biz"), F.col("name_is_fi").alias("t_fi"))
        mergeable = F.col("t_biz") & ~F.col("t_fi") & F.col("t_key").isNotNull()
        emap = (emap.join(tgt, "ent_l1", "left")
                    .withColumn("ent_l2", F.when(mergeable, F.concat(F.lit("N:"), F.md5("t_key"))).otherwise(F.col("ent_l1")))
                    .withColumn("rule", F.when(F.col("ent_l2").startswith("N:"), "business_name")
                                         .when(F.col("xw_to").isNotNull(), "crosswalk_6c_5c").otherwise("identity"))
                    .select("cpty_id", "cpty_id_src", "ent_l0", "ent_l1", "ent_l2", "rule"))
        write_full(emap, "cpty_entity_map")

    em, dim = spark.table(T["cpty_entity_map"]), spark.table(T["cpty_dim"])
    r = em.agg(F.count("*").alias("cpty_ids"), F.countDistinct("ent_l1").alias("entities_l1"),
               F.countDistinct("ent_l2").alias("entities_l2")).toPandas()
    r["l1_vs_ids"] = r.entities_l1 / r.cpty_ids; r["l2_vs_ids"] = r.entities_l2 / r.cpty_ids
    show(r, "Entity consolidation", pct=("l1_vs_ids", "l2_vs_ids"))
    show(dim.agg(F.sum(F.col("is_business").cast("int")).alias("business_like"),
                 F.sum(F.col("name_is_fi").cast("int")).alias("bank_like_names_not_merged"),
                 F.sum(F.col("is_placeholder").cast("int")).alias("placeholder_names_not_merged")).toPandas(),
         "Name classes in the dimension")
    show(em.groupBy("rule", "cpty_id_src").count().orderBy("rule", "cpty_id_src").toPandas(), "Ids by rule and id source")
    big = (em.filter(F.col("rule") == "business_name").groupBy("ent_l2").agg(F.count("*").alias("ids"))
             .orderBy(F.desc("ids")).limit(20)
             .join(dim.filter(F.col("name_key").isNotNull())
                      .select(F.concat(F.lit("N:"), F.md5("name_key")).alias("ent_l2"), "name_key").distinct(), "ent_l2")
             .select("name_key", "ids").orderBy(F.desc("ids")).toPandas())
    show(big, "Largest business-name merges (sanity check)")

## §6 Sensitivity: do per-customer metrics move between `ent_l0` (raw id) and `ent_l2`?

These are computed on the latest `SENS_MONTHS` built months, per customer, month and direction:
- **Breadth:** the number of distinct counterparties.
- **Concentration:** the top counterparty's share of dollars.
- **Churn:** new plus lost counterparties versus the previous month.

High correlations and small shares of "moved" customers mean the raw ids are safe for these metrics.

In [ ]:
if RUN_SENSITIVITY:
    SENS = sorted(m for m in RUN_MONTHS if month_done("edge_monthly", m))[-SENS_MONTHS:]
    with timed("sensitivity"):
        em  = spark.table(T["cpty_entity_map"]).select("cpty_id", "ent_l0", "ent_l2")
        mon = spark.read.parquet(P["edge_monthly"]).filter(F.col("month").isin(SENS) & ~F.col("is_self"))
        x = (mon.filter((F.col("src_kind") == "CUST") & (F.col("dst_kind") == "CPTY"))
                .select("month", F.col("src_id").alias("cust"), F.col("dst_id").alias("cpty"), "amount", F.lit("OUT").alias("dir"))
             .unionByName(mon.filter((F.col("src_kind") == "CPTY") & (F.col("dst_kind") == "CUST"))
                .select("month", F.col("dst_id").alias("cust"), F.col("src_id").alias("cpty"), "amount", F.lit("IN").alias("dir")))
             .filter(F.col("cpty").isNotNull())
             .join(em, F.col("cpty") == F.col("cpty_id"), "left")
             .withColumn("l0", F.coalesce("ent_l0", "cpty")).withColumn("l2", F.coalesce("ent_l2", "cpty"))
             .withColumn("amt", F.col("amount").cast("double"))
             .select("month", "dir", "cust", "l0", "l2", "amt").persist())

        def per_cust(lv):
            g = x.groupBy("month", "dir", "cust", lv).agg(F.sum("amt").alias("a"))
            return g.groupBy("month", "dir", "cust").agg(F.count("*").alias(f"n_{lv}"), (F.max("a") / F.sum("a")).alias(f"top_{lv}"))
        c = per_cust("l0").join(per_cust("l2"), ["month", "dir", "cust"])
        sens = (c.groupBy("dir").agg(
                    F.count("*").alias("customer_months"),
                    F.expr("percentile_approx(n_l0, 0.5)").alias("median_n_l0"),
                    F.expr("percentile_approx(n_l2, 0.5)").alias("median_n_l2"),
                    F.avg((F.col("n_l2") <= 0.9 * F.col("n_l0")).cast("double")).alias("breadth_down_10pct_plus"),
                    F.corr(F.log1p("n_l0"), F.log1p("n_l2")).alias("corr_log_breadth"),
                    F.avg((F.abs(F.col("top_l2") - F.col("top_l0")) > 0.05).cast("double")).alias("top_share_moves_5pp_plus"))
                .toPandas())
        show(sens, f"Breadth and concentration, months {SENS[0]}..{SENS[-1]}", pct=("breadth_down_10pct_plus", "top_share_moves_5pp_plus"))

        if len(SENS) >= 2:
            nxt = lambda col: F.date_format(F.add_months(F.to_date(F.concat(col, F.lit("-01"))), 1), "yyyy-MM")
            def churn(lv):
                e = x.select("month", "dir", "cust", F.col(lv).alias("e")).distinct()
                prev = e.withColumn("month", nxt("month"))              # last month's set, re-labelled as this month
                k = ["month", "dir", "cust", "e"]
                new  = e.join(prev, k, "left_anti").groupBy("month", "dir", "cust").agg(F.count("*").alias("new"))
                lost = prev.join(e, k, "left_anti").groupBy("month", "dir", "cust").agg(F.count("*").alias("lost"))
                return (new.join(lost, ["month", "dir", "cust"], "full").fillna(0)
                           .withColumn(f"churn_{lv}", F.col("new") + F.col("lost")).select("month", "dir", "cust", f"churn_{lv}"))
            ch = (churn("l0").join(churn("l2"), ["month", "dir", "cust"], "full").fillna(0)
                    .filter(F.col("month").isin(SENS[1:])))
            cs = (ch.groupBy("dir").agg(
                      F.count("*").alias("customer_months"),
                      F.expr("percentile_approx(churn_l0, 0.5)").alias("median_churn_l0"),
                      F.expr("percentile_approx(churn_l2, 0.5)").alias("median_churn_l2"),
                      F.avg((F.col("churn_l2") <= 0.9 * F.col("churn_l0")).cast("double")).alias("churn_down_10pct_plus"),
                      F.corr(F.log1p("churn_l0"), F.log1p("churn_l2")).alias("corr_log_churn")).toPandas())
            show(cs, f"Churn (new + lost counterparties vs previous month), {SENS[1]}..{SENS[-1]}", pct=("churn_down_10pct_plus",))
        x.unpersist()

## §7 Impala

The tables are standard external parquet tables, so Impala can query them after its metadata is refreshed. Run the printed
statements in Impala: `INVALIDATE METADATA` the first time a table is created, `REFRESH` after every Spark run.

In [ ]:
for k in KEYS_T:
    print(f"INVALIDATE METADATA {T[k]};   -- first time only")
print()
for k in KEYS_T:
    print(f"REFRESH {T[k]};")
print(f"\n-- optional, faster joins:\nCOMPUTE INCREMENTAL STATS {T['edge_monthly']};")